# Discrete Diffusion Models

**Diffusion models** return us to an explicitly probabilistic style of reasoning, but they do so in a very different way from VAEs. Instead of introducing a single latent code $\boldsymbol{z}$ that explains an image, we introduce an entire latent trajectory $\boldsymbol{x}_0, \boldsymbol{x}_1, \ldots, \boldsymbol{x}_T$, where $\boldsymbol{x}_0$ is the clean data point and the later variables are progressively corrupted versions of it. The key idea is that while direct generation of $\boldsymbol{x}_0$ is difficult, generation may become much easier if we learn to invert a sequence of small corruption steps. Each reverse transition only needs to remove a little bit of noise, and this local denoising problem turns out to be remarkably well suited to neural networks.

In this chapter we focus on the discrete-time viewpoint that leads to denoising diffusion probabilistic models, or DDPMs. Later chapters will reinterpret the same ideas in continuous time using stochastic differential equations. For now, the discrete presentation is valuable because it exposes the relation with latent-variable modeling and makes the training objective look like a structured variational bound.

The transition from GANs to diffusion is one of the most conceptually important turns in the course. GANs showed that excellent samples can arise from an implicit model trained by an adaptive critic, but they also exposed how difficult unstable game optimization can be. Diffusion responds by changing not only the objective, but the granularity of the generative task itself. Instead of asking for one direct leap from noise to image, it asks for many small denoising corrections. This change in decomposition is a large part of why diffusion training often feels much more robust in practice.

There is a useful pedagogical contrast with the VAE chapter. A VAE introduced one latent variable and then spent most of its effort dealing with posterior intractability. A diffusion model takes the opposite path: it introduces a very large number of latent variables, one for each noise level in the chain, but chooses the forward process so carefully that many of the relevant Gaussian quantities become analytically tractable. The resulting model is therefore more elaborate in latent structure but, in some sense, simpler in local conditional algebra. This is one reason diffusion models often feel paradoxical at first: they are conceptually deeper than a vanilla autoencoder, yet some of their core training identities are easier to manipulate once the forward process has been designed.

It is also helpful to say explicitly what problem diffusion is solving compared with GANs. GANs tried to learn a direct path from latent noise to realistic images through a learned adversarial game, and their difficulty lay largely in unstable optimization. Diffusion chooses a much longer route. It asks the model to solve many easy denoising problems instead of one hard synthesis problem. The price is sampling cost. The reward is a training objective that behaves much more like a supervised regression problem. Much of the modern success of diffusion models comes from this tradeoff.

```{figure} ../assets/images/DM.png
:width: 82%
:align: center

A conceptual diagram of the diffusion-model viewpoint: a forward noising path and a learned reverse denoising path.
```

## The Forward Noising Process

Let $\boldsymbol{x}_0 \sim p_{gt}$ be a data sample. We define a forward Markov chain
:::{math}
q(\boldsymbol{x}_{1:T} | \boldsymbol{x}_0)
=
\prod_{t=1}^T q(\boldsymbol{x}_t | \boldsymbol{x}_{t-1}),
:::
where each transition adds a small amount of Gaussian noise:
:::{math}
q(\boldsymbol{x}_t | \boldsymbol{x}_{t-1})
=
\mathcal{N}\big(
    \boldsymbol{x}_t;
    \sqrt{1-\beta_t}\,\boldsymbol{x}_{t-1},
    \beta_t \boldsymbol{I}
\big).
:::
The sequence $\beta_t \in (0,1)$ is called the **noise schedule**. When $\beta_t$ is small, the transition only perturbs the image slightly. Repeating this many times gradually destroys structure until the terminal variable $\boldsymbol{x}_T$ is close to an isotropic Gaussian.

It is helpful to introduce the shorthand $\alpha_t = 1-\beta_t$ and $\overline{\alpha}_t = \prod_{s=1}^t \alpha_s$. These quantities appear constantly because they summarize how much signal survives after many noising steps.

```{prf:theorem} Closed form of the forward marginal
:label: thm-forward-marginal-ddpm

For every $t \in \{1,\ldots,T\}$,
:::{math}
q(\boldsymbol{x}_t | \boldsymbol{x}_0)
=
\mathcal{N}\big(
    \boldsymbol{x}_t;
    \sqrt{\overline{\alpha}_t}\,\boldsymbol{x}_0,
    (1-\overline{\alpha}_t)\boldsymbol{I}
\big).
:::
Equivalently,
:::{math}
\boldsymbol{x}_t
=
\sqrt{\overline{\alpha}_t}\,\boldsymbol{x}_0
+
\sqrt{1-\overline{\alpha}_t}\,\boldsymbol{\varepsilon},
\qquad
\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
```

```{prf:proof}
We proceed by induction on $t$. For $t=1$ the formula is immediate because $\overline{\alpha}_1 = \alpha_1$ and
:::{math}
q(\boldsymbol{x}_1 | \boldsymbol{x}_0)
=
\mathcal{N}\big(
    \boldsymbol{x}_1;
    \sqrt{\alpha_1}\,\boldsymbol{x}_0,
    (1-\alpha_1)\boldsymbol{I}
\big).
:::
Assume the statement holds at time $t-1$, so that
:::{math}
\boldsymbol{x}_{t-1}
=
\sqrt{\overline{\alpha}_{t-1}}\,\boldsymbol{x}_0
+
\sqrt{1-\overline{\alpha}_{t-1}}\,\boldsymbol{\varepsilon}_{t-1}
:::
for a standard Gaussian $\boldsymbol{\varepsilon}_{t-1}$. The next transition gives
:::{math}
\boldsymbol{x}_t
=
\sqrt{\alpha_t}\,\boldsymbol{x}_{t-1}
+
\sqrt{1-\alpha_t}\,\boldsymbol{\eta}_t,
\qquad
\boldsymbol{\eta}_t \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I}).
:::
Substituting the induction hypothesis,
:::{math}
\boldsymbol{x}_t
=
\sqrt{\alpha_t\overline{\alpha}_{t-1}}\,\boldsymbol{x}_0
+
\sqrt{\alpha_t(1-\overline{\alpha}_{t-1})}\,\boldsymbol{\varepsilon}_{t-1}
+
\sqrt{1-\alpha_t}\,\boldsymbol{\eta}_t.
:::
The last two terms are independent centered Gaussians, so their sum is again Gaussian with covariance
:::{math}
\alpha_t(1-\overline{\alpha}_{t-1})\boldsymbol{I}
+
(1-\alpha_t)\boldsymbol{I}
=
(1-\alpha_t\overline{\alpha}_{t-1})\boldsymbol{I}
=
(1-\overline{\alpha}_t)\boldsymbol{I}.
:::
Since $\alpha_t\overline{\alpha}_{t-1} = \overline{\alpha}_t$, the claimed formula follows.
```

This theorem is one of the reasons diffusion models are computationally practical. It says that we do not need to simulate the entire forward chain to sample a noisy observation at time $t$. We can draw $\boldsymbol{x}_t$ directly from $\boldsymbol{x}_0$ in one step. During training, this allows us to select a random time index and generate the corresponding noisy sample without unrolling all earlier transitions.

A concrete image example helps. If $\boldsymbol{x}_0$ is a clean handbag image and $t$ is small, the corresponding $\boldsymbol{x}_t$ still looks like the same handbag with a light dusting of Gaussian corruption. If $t$ is large, the shape is barely recognizable and the sample is close to pure static. Training teaches one network to operate across this whole range of difficulty.

```{admonition} Numerical Example: How Much Signal Survives?
:class: numerical-example

Take a toy noise schedule with $\beta_1 = 0.1$ and $\beta_2 = 0.2$. Then $\alpha_1 = 0.9$, $\alpha_2 = 0.8$, and $\overline{\alpha}_2 = 0.72$. The closed-form forward marginal says
:::{math}
\boldsymbol{x}_2 = \sqrt{0.72}\,\boldsymbol{x}_0 + \sqrt{0.28}\,\boldsymbol{\varepsilon}.
:::
Numerically, $\sqrt{0.72} \approx 0.849$ and $\sqrt{0.28} \approx 0.529$. So after only two steps, about eighty-five percent of the original signal amplitude is still present, but a substantial noise term has already been mixed in.

This is a useful sanity check for intuition. Early diffusion steps do not destroy the image completely. They weaken the signal gradually while increasing uncertainty, which is exactly why the reverse denoiser can hope to recover the structure one small step at a time.
```

Conceptually, the theorem also explains why diffusion training can be randomized over time in such a simple way. The network does not need to see only neighboring states such as $\boldsymbol{x}_{t-1}$ and $\boldsymbol{x}_t$. Because $\boldsymbol{x}_t$ has a direct closed form given $\boldsymbol{x}_0$, the algorithm can choose a random noise level and present the model with the corresponding corrupted example immediately. This makes the training loop look much closer to ordinary supervised learning than one might expect from the long latent trajectory written in the model definition.

For students coming from applied backgrounds, there is a useful interpretation here. The scalar $\overline{\alpha}_t$ measures how much signal survives by time $t$, while $1-\overline{\alpha}_t$ measures how much isotropic Gaussian corruption has been accumulated. Early times correspond to a slightly perturbed image. Late times correspond to something close to pure noise. The network is therefore exposed to a curriculum of denoising subproblems indexed by noise scale.

## The Reverse Model

The forward chain is fixed and easy to sample from. The generative task is to learn a reverse chain
:::{math}
p_\theta(\boldsymbol{x}_{0:T})
=
p(\boldsymbol{x}_T)\prod_{t=1}^T p_\theta(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t),
:::
where $p(\boldsymbol{x}_T)$ is chosen to be a standard Gaussian and each reverse transition is modeled as
:::{math}
p_\theta(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t)
=
\mathcal{N}\big(
    \boldsymbol{x}_{t-1};
    \boldsymbol{\mu}_\theta(\boldsymbol{x}_t, t),
    \boldsymbol{\Sigma}_\theta(\boldsymbol{x}_t, t)
\big).
:::
The neural network is therefore asked to undo one infinitesimal corruption step at a time. This local formulation is crucial. Directly generating a realistic image from white noise in one step is very hard. Predicting how to remove a little amount of Gaussian corruption from a partially structured image is far more manageable.

One way to read this model is as a chain of microscopic decoders. Each reverse conditional takes a slightly too noisy image and tries to move it one step closer to the data manifold. No single step is responsible for creating the whole image. Global structure emerges cumulatively from the repeated composition of local denoising moves. This is very different from the generator in a GAN, which must learn a direct global transport from latent noise to the image space in one pass.

This local perspective is also why U-Nets become such a natural architecture for diffusion. At intermediate times, the input to the network already contains a coarse version of the eventual image, buried under noise. The model does not need to invent all structure from scratch; it needs to preserve what is reliable, infer what is ambiguous, and progressively sharpen detail across scales. Architecturally, that is exactly the sort of task for which multi-scale skip-connected networks are well suited.

## The Posterior of One Forward Step

To derive the training objective, we need the exact Gaussian posterior
:::{math}
q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t, \boldsymbol{x}_0).
:::
Intuitively, if we know both the clean image $\boldsymbol{x}_0$ and the noisy image $\boldsymbol{x}_t$, then the previous state $\boldsymbol{x}_{t-1}$ is no longer mysterious. It is just a Gaussian variable constrained by two linear Gaussian relations. The next theorem gives the exact formula.

```{prf:theorem} Closed form of the one-step posterior
:label: thm-ddpm-posterior

For $t \geq 2$,
:::{math}
q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t, \boldsymbol{x}_0)
=
\mathcal{N}\big(
    \boldsymbol{x}_{t-1};
    \widetilde{\boldsymbol{\mu}}_t(\boldsymbol{x}_t, \boldsymbol{x}_0),
    \widetilde{\beta}_t \boldsymbol{I}
\big),
:::
where
:::{math}
\widetilde{\beta}_t
=
\frac{1-\overline{\alpha}_{t-1}}{1-\overline{\alpha}_t}\beta_t
:::
and
:::{math}
\widetilde{\boldsymbol{\mu}}_t(\boldsymbol{x}_t, \boldsymbol{x}_0)
=
\frac{\sqrt{\overline{\alpha}_{t-1}}\beta_t}{1-\overline{\alpha}_t}\boldsymbol{x}_0
+
\frac{\sqrt{\alpha_t}(1-\overline{\alpha}_{t-1})}{1-\overline{\alpha}_t}\boldsymbol{x}_t.
:::
```

```{prf:proof}
By Bayes' rule and the Markov structure,
:::{math}
q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t, \boldsymbol{x}_0)
\propto
q(\boldsymbol{x}_t | \boldsymbol{x}_{t-1})q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_0).
:::
Both terms are Gaussian in $\boldsymbol{x}_{t-1}$. Writing their log densities and collecting quadratic and linear terms in $\boldsymbol{x}_{t-1}$ yields another Gaussian density. The precision matrix is scalar times the identity, so only scalar coefficients need to be combined. Completing the square gives posterior variance
:::{math}
\widetilde{\beta}_t
=
\frac{1-\overline{\alpha}_{t-1}}{1-\overline{\alpha}_t}\beta_t
:::
and posterior mean
:::{math}
\widetilde{\boldsymbol{\mu}}_t(\boldsymbol{x}_t, \boldsymbol{x}_0)
=
\frac{\sqrt{\overline{\alpha}_{t-1}}\beta_t}{1-\overline{\alpha}_t}\boldsymbol{x}_0
+
\frac{\sqrt{\alpha_t}(1-\overline{\alpha}_{t-1})}{1-\overline{\alpha}_t}\boldsymbol{x}_t.
:::
The calculation is straightforward but algebraically long, and its importance lies in the conclusion: the reverse posterior is Gaussian with parameters known in closed form once $\boldsymbol{x}_0$ is available.
```

This posterior formula makes diffusion models look strikingly close to latent-variable models. The forward chain defines a tractable encoder-like distribution $q$. The reverse chain defines a learned decoder-like distribution $p_\theta$. The main difference from a VAE is that the latent structure is not a single vector but a long Markov chain whose conditionals have been chosen to admit closed-form Gaussian manipulations.

It is worth stressing how unusual this design choice is. In a VAE, the variational distribution is learned because the exact posterior is too hard to compute. In DDPMs, much of the posterior structure is known analytically because the forward corruption was fixed in advance. The modeling freedom is moved away from the encoder side and concentrated in the reverse denoiser. This is one of the deepest structural differences between the two families, even though both are trained through an ELBO.

The theorem also clarifies why the reverse model is asked to predict relatively simple Gaussian statistics rather than an arbitrary conditional density. Once the forward chain is chosen, the true reverse posterior conditioned on $\boldsymbol{x}_0$ is already Gaussian. The learning problem becomes: can a neural network infer the right Gaussian mean, or an equivalent target such as the added noise, from the noisy image and the time index alone?

## Variational Derivation of the DDPM Objective

The log-likelihood of the model can be lower bounded using the forward process as a variational distribution. Starting from
:::{math}
\log p_\theta(\boldsymbol{x}_0)
=
\log \int p_\theta(\boldsymbol{x}_{0:T})\, d\boldsymbol{x}_{1:T},
:::
we multiply and divide by $q(\boldsymbol{x}_{1:T} | \boldsymbol{x}_0)$ and apply Jensen's inequality exactly as in the VAE derivation. This yields the evidence lower bound
:::{math}
\log p_\theta(\boldsymbol{x}_0)
\geq
\mathbb{E}_{q(\boldsymbol{x}_{1:T} | \boldsymbol{x}_0)}
\left[
    \log p_\theta(\boldsymbol{x}_{0:T})
    -
    \log q(\boldsymbol{x}_{1:T} | \boldsymbol{x}_0)
\right].
:::
After expanding the Markov factorizations and regrouping terms, one obtains a sum of Kullback-Leibler divergences between exact forward posteriors and learned reverse transitions, plus a terminal prior matching term and a reconstruction term at time $t=1$.

```{prf:theorem} ELBO decomposition for discrete diffusion
:label: thm-ddpm-elbo

Up to sign conventions, the negative ELBO can be decomposed as
:::{math}
\mathcal{L}
=
\mathbb{E}_q\big[
    \operatorname{KL}(q(\boldsymbol{x}_T | \boldsymbol{x}_0) \| p(\boldsymbol{x}_T))
    +
    \sum_{t=2}^T
    \operatorname{KL}\big(
        q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t, \boldsymbol{x}_0)
        \|
        p_\theta(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t)
    \big)
    - \log p_\theta(\boldsymbol{x}_0 | \boldsymbol{x}_1)
\big].
:::
```

```{prf:proof}
Expand the ELBO integrand using
:::{math}
p_\theta(\boldsymbol{x}_{0:T})
=
p(\boldsymbol{x}_T)\prod_{t=1}^T p_\theta(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t)
:::
and
:::{math}
q(\boldsymbol{x}_{1:T} | \boldsymbol{x}_0)
=
\prod_{t=1}^T q(\boldsymbol{x}_t | \boldsymbol{x}_{t-1}).
:::
Rearranging the resulting sum and repeatedly inserting
:::{math}
q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_t, \boldsymbol{x}_0)
=
\frac{q(\boldsymbol{x}_t | \boldsymbol{x}_{t-1})q(\boldsymbol{x}_{t-1} | \boldsymbol{x}_0)}
     {q(\boldsymbol{x}_t | \boldsymbol{x}_0)}
:::
produces one KL term for each reverse step, one KL term matching the terminal marginal to the chosen Gaussian prior, and one data reconstruction term involving $p_\theta(\boldsymbol{x}_0 | \boldsymbol{x}_1)$. The proof is structurally the same as in variational latent-variable models, except that the latent space is now the entire trajectory.
```

The importance of this theorem is conceptual. A DDPM can be viewed as a very deep latent-variable model whose latent variables are the noisy intermediate states. The forward process plays the role of an analytically chosen inference model, while the reverse chain is learned. This is precisely why the connection with VAEs is not superficial. Both methods optimize a variational lower bound. They differ mainly in how the latent structure is designed and in how expressive the reverse conditionals become when the number of steps is large.

This VAE connection deserves to be made even more explicit. If one writes the VAE ELBO schematically, one sees a reconstruction term plus a regularization term that compares an approximate posterior with a prior. In diffusion, the same broad logic survives, but it is distributed across time. Each KL term compares one exact local reverse posterior from the forward process with one learned local reverse conditional. Instead of asking one latent variable to summarize the whole image, the model asks a long chain of noisy latents to carry the information gradually. The gain is that each local denoising task is simple. The cost is that generation becomes iterative.

From a course-design perspective, this is a valuable moment because it unifies two chapters that may otherwise feel unrelated. VAEs and DDPMs are not competing only at the level of sample quality. They represent two different choices about where to place tractability. VAEs keep sampling cheap but inference approximate. Diffusion keeps local posterior algebra easy but sampling long. Understanding this tradeoff helps students see why modern generative modeling is full of architectures that redistribute difficulty rather than eliminate it.

## Noise Prediction Parameterization

The reverse mean can be parameterized in several equivalent ways. One especially successful choice is to make the neural network predict the noise $\boldsymbol{\varepsilon}$ used to generate $\boldsymbol{x}_t$ from $\boldsymbol{x}_0$. From the forward marginal identity,
:::{math}
\boldsymbol{x}_t
=
\sqrt{\overline{\alpha}_t}\boldsymbol{x}_0
+
\sqrt{1-\overline{\alpha}_t}\boldsymbol{\varepsilon},
:::
we can solve for $\boldsymbol{x}_0$:
:::{math}
\boldsymbol{x}_0
=
\frac{1}{\sqrt{\overline{\alpha}_t}}
\left(
    \boldsymbol{x}_t
    -
    \sqrt{1-\overline{\alpha}_t}\,\boldsymbol{\varepsilon}
\right).
:::
If a network $\boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t, t)$ predicts the noise, then this estimate can be substituted into the posterior mean formula. After fixing the reverse variance to a known schedule, the corresponding KL term reduces to a weighted mean squared error between the true noise and the predicted noise.

In many practical expositions, the training objective is therefore written in the simplified form
:::{math}
\mathcal{L}_{simple}(\theta)
=
\mathbb{E}_{t,\boldsymbol{x}_0,\boldsymbol{\varepsilon}}
\left[
    \|
        \boldsymbol{\varepsilon}
        -
        \boldsymbol{\varepsilon}_\theta(\boldsymbol{x}_t, t)
    \|_2^2
\right],
:::
where
:::{math}
\boldsymbol{x}_t
=
\sqrt{\overline{\alpha}_t}\boldsymbol{x}_0
+
\sqrt{1-\overline{\alpha}_t}\boldsymbol{\varepsilon}.
:::
This is one of the most elegant features of DDPMs. A complicated variational objective turns into a denoising regression problem that is easy to implement and optimize. Later, when we discuss score-based models, we shall see that predicting the noise is closely related to predicting the score of the perturbed data distribution.

The simplification above is not merely a coding convenience. It changes the way one should interpret the method. The network is no longer viewed primarily as a density model in the classical sense. It becomes a time-conditioned denoiser trained across many corruption scales. The probabilistic interpretation remains essential, because it explains why this denoising regression objective is legitimate, but the optimization face of the model is strikingly close to supervised learning on synthetic noisy targets.

This is one reason diffusion models were comparatively easy to scale once the architectural ingredients became available. Deep learning already knows how to fit large regression networks well. If one can reformulate generative modeling so that the central neural task is "predict the noise that was added," a great deal of existing optimization machinery becomes immediately useful.

## DDIM and Deterministic Sampling

The original DDPM sampler is stochastic because each reverse step samples from a Gaussian conditional. A natural question is whether one can keep the same training objective but sample along a more deterministic path. Denoising diffusion implicit models, or DDIMs, answer yes. They reinterpret the same learned denoiser through a non-Markovian construction whose marginals match the training family while permitting deterministic or partially stochastic reverse trajectories.

The conceptual value of DDIM is very high. It shows that the learned network is not tied to one unique reverse chain. Once the model has learned how noisy observations relate to clean structure across time, one can choose different numerical trajectories for generation. Deterministic DDIM sampling often uses fewer steps than the original stochastic DDPM sampler while preserving much of the sample quality. This matters for practice because one of the main weaknesses of diffusion is slow generation.

For students, DDIM is also the first strong hint that diffusion models are closely related to continuous transport viewpoints. The denoiser is trained from one probabilistic construction, yet can be used along another trajectory that looks much more like solving an evolution equation than repeatedly sampling fresh randomness. This observation will become even clearer in the continuous-time chapter when probability flow ODEs enter the picture.

## Relation with VAEs, Latent Diffusion, and Preview of the Continuous-Time Limit

It is worth closing this discrete chapter by clarifying the relation with the models studied earlier. A VAE learns a decoder from a compact latent code and must approximate the posterior because exact inference is hard. A discrete diffusion model chooses a much more elaborate latent structure, but in exchange the forward corruption process is analytically fixed and its marginals and local posteriors are tractable. The result is an optimization problem that remains variational in spirit while being unusually stable in practice.

Modern latent diffusion models add another twist to this story. Instead of running diffusion directly in pixel space, one first compresses images into a lower-dimensional latent representation using an autoencoding model, and then performs diffusion in that latent space. The motivation is computational rather than philosophical: denoising a compressed representation is dramatically cheaper than denoising full-resolution pixels, especially for high-resolution image synthesis. In this sense, latent diffusion can be viewed as a reconciliation between autoencoding ideas and diffusion ideas. The autoencoder provides a perceptually meaningful lower-dimensional space, and the diffusion model supplies the powerful generative prior inside that space.

The price paid by diffusion models is therefore partly relocated rather than removed. Standard pixel-space diffusion is stable but slow. Latent diffusion makes sampling cheaper and scales better to large images, but it inherits the quality and inductive biases of the underlying latent representation. Understanding these tradeoffs is important because many practical text-to-image systems are best thought of not as raw DDPMs, but as carefully engineered latent diffusion pipelines.

Finally, understanding what happens when the number of diffusion steps becomes very large leads naturally to continuous-time stochastic dynamics. That is the purpose of the next chapter, where the discrete denoising view will be reinterpreted through score matching, SDEs, and probability flow ODEs. The foundational DDPM perspective comes from {cite}`sohl2015deep` and {cite}`ho2020denoising`, while the deterministic sampling perspective and latent-space scaling directions are represented by {cite}`song2020ddim` and {cite}`rombach2022high`.

```{figure} ../assets/images/diffusion-diagram.png
:width: 82%
:align: center

A conceptual picture of the diffusion process: data are gradually corrupted in the forward direction and progressively denoised in the learned reverse direction.
```